# Selection visualization notebook
This notebook creates an alluvial diagram that shows how rows from gold standard tables flow into result tables for a selected database and evaluation result set.


In [30]:
# Import required libraries
import json
from collections import defaultdict, Counter
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

from util.evaluation.f1_score import F1Score, EvaluationType


In [31]:
# Load Gold Standard and Results Data
root = Path("data")
experiment_name = "batch_insertion"
result_file_name = "evaluation_results_0.8_0.2_0.375_0.5_0.5.json"

available_databases = sorted([
    path.name
    for path in root.iterdir()
    if path.is_dir() and (path / "gold_standard_results.json").exists() # and path.name.startswith("spider2")
])

if not available_databases:
    raise FileNotFoundError("No databases with gold_standard_results.json found in data/")

print("Available databases:")
for name in available_databases:
    print(" -", name)


available_strategies = ["justine_3_3_one_prompt", "justine_3_3_retry_with_feedback"]
available_strategies = sorted([
    path.name
    for path in (root / available_databases[0] / experiment_name).iterdir()
    if path.is_dir() and path.name.startswith("justine_3_3_")
])

f1_score = F1Score(include_null_values=True, strict_score=True)
precision = F1Score(evaluation_type=EvaluationType.PRECISION, include_null_values=True, strict_score=True)
recall = F1Score(evaluation_type=EvaluationType.RECALL, include_null_values=True, strict_score=True)


Available databases:
 - bird_california_schools
 - bird_card_games
 - bird_codebase_community
 - bird_debit_card_specializing
 - bird_european_football_2
 - bird_financial
 - bird_formula_1
 - bird_student_club
 - bird_superhero
 - bird_thrombosis_prediction
 - bird_toxicology
 - spider2_Airlines
 - spider2_California_Traffic_Collision
 - spider2_Db-IMDB
 - spider2_EU_soccer
 - spider2_EntertainmentAgency
 - spider2_IPL
 - spider2_Pagila
 - spider2_imdb_movies
 - spider2_school_scheduling
 - spider2_stacking
 - spider_candidate_poll
 - spider_chinook_1
 - spider_company_office
 - spider_decoration_competition
 - spider_entrepreneur
 - spider_soccer_2
 - spider_swimming
 - spider_theme_gallery
 - spider_workshop_paper
 - wikidb_alpha-defensins
 - wikidb_burial-vault
 - wikidb_chemotaxis-methyl-accepting-receptor-tar-related-ligand-binding-domain-protein-family
 - wikidb_heaphy-sketchbook-no-2
 - wikidb_japans-top-100-waterfalls
 - wikidb_novelette
 - wikidb_paz-domain-superfamily
 - wik

In [32]:
def is_correct_row(
    row_1: list[str],
    row_2: list[str],
) -> bool:
    """Checks if both rows contain the same values"""
    c1 = Counter([gs_value for gs_value in row_1 if f1_score._consider_value(gs_value)])
    c2 = Counter([r_value for r_value in row_2 if f1_score._consider_value(r_value)])
    return c1 == c2

def is_row_in_result_table(gold_row, result_table):
    exists = any(
        is_correct_row(gold_row, result_row)
        for result_row in result_table
    )
    return exists

def is_value_in_result_column(gold_row, gold_column_index, result_table, result_column_index):
    exists = any(
        gold_row[gold_column_index] == result_row[result_column_index] and is_correct_row(gold_row, result_row)
        for result_row in result_table
    )
    return exists

def find_result_file(gold_db_name: str, strategy_name: str) -> Path:
    result_dir = root / gold_db_name / experiment_name / strategy_name
    if not result_dir.exists():
        raise FileNotFoundError(f"Result directory not found: {result_dir}.")

    preferred_path = result_dir / result_file_name
    if preferred_path.exists():
        return preferred_path

    candidates = sorted(result_dir.glob("*.json"))
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"Could not find a JSON result file in {result_dir}."
    )

def load_database_and_result(gold_db_name: str, strategy_name: str):
    gold_path = root / gold_db_name / "gold_standard_results.json"
    if not gold_path.exists():
        raise FileNotFoundError(f"Could not find gold standard file at {gold_path}")

    result_path = find_result_file(gold_db_name, strategy_name)

    with open(gold_path, encoding="utf-8") as f:
        gold_standard = json.load(f)

    try:
        with open(result_path, encoding="utf-8") as f:
            result_database = json.load(f)
    except:
        result_database = {}

    return gold_standard, result_database, result_path

def map_gold_rows_to_result_tables(gold_standard, result_database):
    result_database = {table_name: [[f1_score._normalize(value) for value in row] for row in table] for table_name, table in result_database.items()}
    gold_standard = {table_name: [[f1_score._normalize(value) for value in row] for row in table] for table_name, table in gold_standard.items()}

    mapping = defaultdict(int)

    for gold_table_name, gold_rows in gold_standard.items():    
        gold_table_name = "gs: " + gold_table_name[14:-1]
        for gold_row in gold_rows:
            matched_tables = [
                result_table_name[14:-1]
                for result_table_name, result_rows in result_database.items()
                if is_row_in_result_table(gold_row, result_rows)
            ]

            if matched_tables:
                for result_table_name in matched_tables:
                    mapping[(gold_table_name, result_table_name)] += 1
            else:
                mapping[(gold_table_name, "NO_MATCH")] += 1

    return mapping


In [ ]:
# Prepare and render alluvial diagram

def prepare_sankey_data(row_mapping):
    link_rows = []
    for (gold_table, result_table), count in sorted(
        row_mapping.items(), key=lambda x: x[1], reverse=True
    ):
        if count <= 0:
            continue
        link_rows.append({
            "gold_table": gold_table,
            "result_table": result_table,
            "count": count,
        })

    flow_df = pd.DataFrame(link_rows)
    labels = list(pd.unique(flow_df[["gold_table", "result_table"]].values.ravel("K")))
    label_to_id = {label: idx for idx, label in enumerate(labels)}

    flow_df["source"] = flow_df["gold_table"].map(label_to_id)
    flow_df["target"] = flow_df["result_table"].map(label_to_id)

    return flow_df, labels


def make_sankey_figure(flow_df, labels, title, custom_node_data=None, custom_link_data=None):
    link_colors = [
        "rgba(150, 200, 250, 0.6)"
        if target_label != "NO_MATCH"
        else "rgba(200, 100, 100, 0.6)"
        for target_label in flow_df["target_label"]
    ]
    node_colors = [
        "rgba(25, 50, 200, 0.8)"
        if "NO_MATCH" not in label
        else "rgba(180, 80, 80, 0.8)"
        for label in labels
    ]

    node_kwargs = {
        "pad": 20,
        "thickness": 20,
        "line": {"color": "black", "width": 0.5},
        "label": labels,
        "color": node_colors,
    }
    if custom_node_data is not None:
        node_kwargs["customdata"] = custom_node_data
        node_kwargs["hovertemplate"] = (
            "<b>%{label}</b><br>"
            "Total connections: %{value} rows<br>"
            "F1-Score: %{customdata[0]}<br>"
            "Precision: %{customdata[1]}<br>"
            "Recall: %{customdata[2]}"
            "<extra></extra>"
        )

    link_kwargs = {
        "source": flow_df["source"].tolist(),
        "target": flow_df["target"].tolist(),
        "value": flow_df["count"].tolist(),
        "color": link_colors,
        "hovertemplate": (
            "%{source.label} → %{target.label}: %{value} rows<extra></extra>"
        ),
    }
    if custom_link_data is not None:
        link_kwargs["customdata"] = custom_link_data
        link_kwargs["hovertemplate"] = (
            "%{source.label} → %{target.label}: %{value} rows<br>"
            "F1-Score: %{customdata[0]}<br>"
            "Precision: %{customdata[1]}<br>"
            "Recall: %{customdata[2]}"
            "<extra></extra>"
        )

    fig = go.Figure(
        data=[
            go.Sankey(
                node=node_kwargs,
                link=link_kwargs
            )
        ]
    )

    fig.update_layout(
        title_text=title,
        font_size=12,
        height=700,
    )
    return fig


def plot_sankey(flow_df, labels, gold_db_name, strategy_name, custom_node_data):
    fig = make_sankey_figure(
        flow_df.assign(target_label=flow_df["result_table"]),
        labels,
        f"Gold standard rows to result tables in {gold_db_name} ({strategy_name})",
        custom_node_data,
    )
    fig.show()


def render_database(gold_db_name, strategy_name):
    gold_standard, result_database, result_path = load_database_and_result(gold_db_name, strategy_name)
    row_mapping = map_gold_rows_to_result_tables(gold_standard, result_database)
    flow_df, labels = prepare_sankey_data(row_mapping)

    custom_node_data = []
    for label in labels:
        if label.startswith("gs: "):
            gs_table = gold_standard[f"SELECT * FROM {label[4:]};"]
            result_db_with_keys = {table_name: (table, [f1_score._row_key(row) for row in table]) for table_name, table in result_database.items()}
            table_f1_score = f1_score._calculate_gs_table_score(gs_table, result_db_with_keys)
            table_precision = precision._calculate_gs_table_score(gs_table, result_db_with_keys)
            table_recall = recall._calculate_gs_table_score(gs_table, result_db_with_keys)
        else:
            table_f1_score, table_precision, table_recall = 0, 0, 0
        custom_node_data.append([table_f1_score, table_precision, table_recall])

    print(f"Loaded {len(gold_standard)} gold tables and {len(result_database)} result tables.")
    print(f"Found {len(row_mapping)} unique gold->result mappings.")
    if flow_df.empty:
        print("No flows were detected for this database and result set.")
        return

    # print("Top flows:")
    # display(flow_df.sort_values("count", ascending=False).head(10))
    # print(f"\nF1-Score: {f1_score.calculate(result_database, gold_standard):.2f}, Precision: {precision.calculate(result_database, gold_standard):.2f}, Recall: {recall.calculate(result_database, gold_standard):.2f}")
    plot_sankey(flow_df, labels, gold_db_name, strategy_name, custom_node_data)


def prepare_column_mapping_data(gold_standard, result_database, gold_table_name):
    result_database = {table_name: [[f1_score._normalize(value) for value in row] for row in table] for table_name, table in result_database.items()}
    gold_standard = {table_name: [[f1_score._normalize(value) for value in row] for row in table] for table_name, table in gold_standard.items()}
    
    gold_table_name = f"SELECT * FROM {gold_table_name};"
    if gold_table_name not in gold_standard:
        raise KeyError(f"Gold table {gold_table_name} not found in the gold standard database.")

    gold_table = gold_standard[gold_table_name]
    gold_column_indices = [
        index
        for index in range(len(gold_table[0]) if len(gold_table) > 0 else 0)
        if any(f1_score._consider_value(row[index]) for row in gold_table)
    ]

    if not gold_column_indices:
        return pd.DataFrame(columns=["source", "target", "count", "target_label"]), []

    link_rows = []
    for gold_column_index in gold_column_indices:
        source_label = f"gs: {gold_table_name[14:-1]} | col {gold_column_index}"
        matched_any = False
        for result_table_name, result_rows in result_database.items():
            result_table_name = result_table_name[14:-1]
            for result_column_index in range(len(result_rows[0]) if len(result_rows) > 0 else 0):
                if not any(f1_score._consider_value(row[result_column_index]) for row in result_rows):
                    continue

                matched_values = [
                    gs_row[gold_column_index]
                    for gs_row in gold_table
                    if f1_score._consider_value(gs_row[gold_column_index])
                    and is_value_in_result_column(
                        gs_row,
                        gold_column_index,
                        result_rows,
                        result_column_index,
                    )
                ]
                if matched_values:
                    link_rows.append({
                        "source": source_label,
                        "target": f"{result_table_name} | col {result_column_index}",
                        "count": len(matched_values),
                        "target_label": f"{result_table_name} | col {result_column_index}",
                    })
                    matched_any = True

        if not matched_any:
            link_rows.append({
                "source": source_label,
                "target": "NO_MATCH",
                "count": 1,
                "target_label": "NO_MATCH",
            })

    if not link_rows:
        return pd.DataFrame(columns=["source", "target", "count", "target_label"]), []

    flow_df = pd.DataFrame(link_rows)
    labels = list(pd.unique(flow_df[["source", "target"]].values.ravel("K")))
    label_to_id = {label: idx for idx, label in enumerate(labels)}

    flow_df["source_id"] = flow_df["source"].map(label_to_id)
    flow_df["target_id"] = flow_df["target"].map(label_to_id)

    return flow_df, labels


def plot_column_mapping(flow_df, labels, gold_db_name, strategy_name, gold_table_name, custom_node_data, custom_link_data):
    if flow_df.empty:
        print("No column-level mappings were detected for this table.")
        return

    fig = make_sankey_figure(
        flow_df.assign(source=flow_df["source_id"], target=flow_df["target_id"], target_label=flow_df["target_label"]),
        labels,
        f"Gold column mapping for {gold_table_name} in {gold_db_name} ({strategy_name})",
        custom_node_data,
        custom_link_data,
    )
    fig.show()


def render_column_mapping(gold_db_name, strategy_name, gold_table_name):
    gold_standard, result_database, _ = load_database_and_result(gold_db_name, strategy_name)
    flow_df, labels = prepare_column_mapping_data(gold_standard, result_database, gold_table_name)

    result_database = {table_name: (table, [f1_score._row_key(row) for row in table]) for table_name, table in result_database.items()}

    custom_node_data = []
    for i, label in enumerate(labels):
        if label.startswith("gs: "):
            gs_table = gold_standard[f"SELECT * FROM {label.split()[1]};"]
            column_f1_score = f1_score._calculate_gs_column_score(i, gs_table, result_database)
            column_precision = precision._calculate_gs_column_score(i, gs_table, result_database)
            column_recall = recall._calculate_gs_column_score(i, gs_table, result_database)
        else:
            column_f1_score, column_precision, column_recall = 0, 0, 0
        custom_node_data.append([column_f1_score, column_precision, column_recall])

    custom_link_data = []
    for source_label, target_label in zip(flow_df["source"].tolist(), flow_df["target"].tolist()):
        if target_label == "NO_MATCH":
            custom_link_data.append([0, 0, 0])
            continue
        gs_column_index = int(source_label.split()[-1])
        gs_table = gold_standard[f"SELECT * FROM {source_label.split()[1]};"]
        result_column_index = int(target_label.split()[-1])
        result_table, result_keys = result_database[f"SELECT * FROM {target_label.split()[0]};"]
        link_f1_score = f1_score._calculate_gs_r_column_score(gs_column_index, gs_table, result_column_index, result_table, result_keys)
        link_precision = precision._calculate_gs_r_column_score(gs_column_index, gs_table, result_column_index, result_table, result_keys)
        link_recall = recall._calculate_gs_r_column_score(gs_column_index, gs_table, result_column_index, result_table, result_keys)
        custom_link_data.append([link_f1_score, link_precision, link_recall])

    if flow_df.empty:
        print(f"No usable columns found for {gold_table_name}.")
        return

    # print(f"Preparing column-level mapping for {gold_table_name}.")
    plot_column_mapping(flow_df, labels, gold_db_name, strategy_name, gold_table_name, custom_node_data, custom_link_data)


In [35]:
# Interactive database and strategy selection and visualization

db_dropdown = widgets.Dropdown(
    options=available_databases,
    value=available_databases[11],
    description="Database:",
    layout=widgets.Layout(width="80%"),
)

strategy_dropdown = widgets.Dropdown(
    options=available_strategies,
    value=available_strategies[0],
    description="Strategy:",
    layout=widgets.Layout(width="80%"),
)

output = widgets.Output()
column_output = widgets.Output()


def get_available_tables(gold_db_name, strategy_name):
    gold_standard, _, _ = load_database_and_result(gold_db_name, strategy_name)
    return sorted([table_name[14:-1] for table_name in gold_standard.keys()])


available_tables = get_available_tables(available_databases[11], available_strategies[0])
table_dropdown = widgets.Dropdown(
    options=available_tables,
    value=available_tables[3],
    description="Gold table:",
    layout=widgets.Layout(width="80%"),
)


def refresh_table_dropdown():
    try:
        refreshed_tables = get_available_tables(db_dropdown.value, strategy_dropdown.value)
        table_dropdown.options = refreshed_tables
        if table_dropdown.value not in refreshed_tables:
            table_dropdown.value = refreshed_tables[3]
    except FileNotFoundError as exc:
        print(exc)


def update_table_visualizations():
    with output:
        clear_output(wait=True)
        try:
            render_database(db_dropdown.value, strategy_dropdown.value)
        except FileNotFoundError as exc:
            print(exc)

def update_column_visualizations():
    with column_output:
        clear_output(wait=True)
        try:
            render_column_mapping(db_dropdown.value, strategy_dropdown.value, table_dropdown.value)
        except FileNotFoundError as exc:
            print(exc)


def on_selection_change(change):
    if change["type"] == "change" and change["name"] == "value":
        if change.get("owner") in {db_dropdown, strategy_dropdown}:
            refresh_table_dropdown()
            update_table_visualizations()
        else:
            update_column_visualizations()


db_dropdown.observe(on_selection_change)
strategy_dropdown.observe(on_selection_change)
table_dropdown.observe(on_selection_change)

display(widgets.VBox([db_dropdown, strategy_dropdown]))
display(output)
display(table_dropdown)
display(column_output)

refresh_table_dropdown()
update_table_visualizations()
update_column_visualizations()

Output()

Dropdown(description='Gold table:', index=3, layout=Layout(width='80%'), options=('aircrafts_data', 'airports_…

Output()